# 第12章 小干扰稳定分析 - SMIB 系统

> Kundur P. *Power System Stability and Control*, McGraw-Hill, 1994, Chapter 12
> Examples 12.1 - 12.4

## 本章概述

本章使用 **Heffron-Phillips 模型**（K1-K6 常数）分析单机无穷大系统的小干扰稳定性。

### Heffron-Phillips K-常数

将 SMIB 系统在工作点附近线性化，得到 6 个常数：

| 常数 | 定义 | 物理意义 |
|------|------|---------|
| K1 | $\\partial P_e/\\partial \\delta$ | 同步转矩系数 |
| K2 | $\\partial P_e/\\partial E_q'$ | 磁场磁链对电磁转矩的影响 |
| K3 | 阻抗系数 | 外部电抗对励磁回路的影响 |
| K4 | $\\partial E_q/\\partial \\delta$ | 功角变化的去磁效应 |
| K5 | $\\partial V_t/\\partial \\delta$ | 功角变化对端电压的影响 |
| K6 | $\\partial V_t/\\partial E_q'$ | 磁链变化对端电压的影响 |

**关键结论：** 在高功角下，K5 < 0，AVR 通过 K5 通道引入负阻尼，
需要 PSS 提供附加正阻尼信号。

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from psa4teaching.stability.heffron_phillips import (
    compute_heffron_phillips_constants, sweep_k_constants, HeffronPhillipsResult
)
from psa4teaching.models import IEEET1Params

plt.rcParams['font.sans-serif'] = ['SimHei', 'Microsoft YaHei', 'DejaVu Sans']
plt.rcParams['axes.unicode_minus'] = False
print('Import OK')

---
## Example 12.1: K1-K6 常数计算

**系统参数 (Kundur Sec. 12.3):**
- 发电机: $X_d=1.8, X_d'=0.3, X_q=1.7, T_{d0}'=8.0$ s, $H=5.0$ s
- 变压器+线路等值电抗: $X_e=0.5$ pu
- 运行点: $E'=1.2, V_\\infty=1.0, \\delta_0=30°$

In [ ]:
# ===== Ex 12.1: K1-K6 常数计算 =====
result = compute_heffron_phillips_constants(
    E_prime=1.2, V_infinity=1.0, X_total=0.5,
    Xd=1.8, Xd_prime=0.3, Xq=1.7,
    Td0_prime=8.0, H=5.0, D=0.0,
    delta_0=np.radians(30), verbose=True
)

# Print K constant summary
print('\nK-常数物理意义:')
print(f'  K1={result.K1:+.4f} -> 同步转矩充足' if result.K1>0 else '  K1<0 -> 同步转矩不足!')
print(f'  K5={result.K5:+.4f} -> {"负阻尼风险! 需要PSS" if result.K5<0 else "正阻尼"}')

---
## Example 12.2: K-常数随功角变化

扫描不同运行功角下的 K1-K6 常数，观察：
1. K1（同步转矩）在 $\\delta=90°$ 处过零
2. K5 在高功角下变负（负阻尼机制的核心）

In [ ]:
# ===== Ex 12.2: K 常数扫描 =====
curves = sweep_k_constants(
    E_prime=1.2, V_infinity=1.0, X_total=0.5,
    delta_range=(5, 85), n_points=81,
    Xd=1.8, Xd_prime=0.3, Xq=1.7
)

fig, axes = plt.subplots(2, 3, figsize=(16, 10))

titles = ['K1 = dPe/ddelta', 'K2 = dPe/dEq\'', 'K3 (field impedance)',
          'K4 = dEq/ddelta', 'K5 = dVt/ddelta', 'K6 = dVt/dEq\'']
keys = ['K1','K2','K3','K4','K5','K6']
colors = ['blue','red','green','purple','orange','brown']

for ax, title, key, color in zip(axes.flat, titles, keys, colors):
    ax.plot(curves['delta_deg'], curves[key], color=color, linewidth=2)
    ax.axhline(y=0, color='k', linewidth=0.5, linestyle='--')
    ax.set_xlabel('delta (deg)')
    ax.set_ylabel(key)
    ax.set_title(title, fontsize=11)
    ax.grid(True, alpha=0.3)

plt.suptitle('K1-K6 vs Rotor Angle (Heffron-Phillips Constants)', fontsize=14, y=1.01)
plt.tight_layout()
plt.savefig('examples/kundur/k_constants_sweep.png', dpi=150, bbox_inches='tight')
plt.show()
print('K1 在 delta≈90° 处过零 -> 静态稳定极限')
print('K5 在高功角下变负 -> AVR 引入负阻尼')

---
## Example 12.3: AVR 增益对阻尼的影响

扫描励磁增益 $K_A$，绘制特征值的根轨迹。

**关键现象：** $K_A$ 增大使励磁模式阻尼增加，但机电模式阻尼减小（通过负的 K5 通道）。

In [ ]:
# ===== Ex 12.3: AVR 增益根轨迹 =====
KA_values = np.logspace(0, 3, 40)  # 1 to 1000

electromech_eig = []  # 机电模式特征值
exciter_eig = []      # 励磁模式特征值

for Ka in KA_values:
    r = compute_heffron_phillips_constants(
        E_prime=1.2, V_infinity=1.0, X_total=0.5,
        Xd=1.8, Xd_prime=0.3, Xq=1.7,
        Td0_prime=8.0, H=5.0, D=0.0,
        delta_0=np.radians(30), Ka=Ka, Te=0.5, verbose=False
    )
    # 分类：低频 (< 5 Hz) = 机电模式, 高频 = 励磁模式
    for ev, f in zip(r.eigenvalues, r.frequencies):
        if abs(ev.imag) > 1e-6:
            if f < 5.0:
                electromech_eig.append((Ka, ev))
            else:
                exciter_eig.append((Ka, ev))

# 绘图
fig, ax = plt.subplots(figsize=(10, 8))
for ka_list, color, label in [(electromech_eig, 'blue', 'Electromech'),
                                 (exciter_eig, 'red', 'Exciter')]:
    if ka_list:
        kas, evs = zip(*ka_list)
        evs = np.array(evs)
        ax.scatter(evs.real, evs.imag, c=np.log10(kas),
                  cmap='viridis', s=30, alpha=0.7, label=label)

ax.axvline(x=0, color='k', linewidth=0.5, linestyle='--')
ax.axhline(y=0, color='k', linewidth=0.5)
ax.set_xlabel('Real part (1/s)')
ax.set_ylabel('Imag part (rad/s)')
ax.set_title('Root Locus: AVR Gain KA sweep (1 to 1000)')
ax.grid(True, alpha=0.3)
ax.legend()
cbar = plt.colorbar(ax.collections[0], ax=ax, label='log10(KA)')
plt.tight_layout()
plt.savefig('examples/kundur/avr_root_locus.png', dpi=150, bbox_inches='tight')
plt.show()
print('随 KA 增大: 机电模式向右侧移动 -> 阻尼减小')
print('负阻尼风险来自 K5 < 0 和 AVR 的相位滞后')

---
## Example 12.4: PSS 参数设计

电力系统稳定器 (PSS) 通过相位补偿抵消 AVR 的负阻尼效应。

**PSS 设计步骤：**
1. 计算无 PSS 时的机电模式特征值
2. 确定需要补偿的相位滞后
3. 设计超前-滞后环节 $(1+sT_1)/(1+sT_2)$ 的参数

典型 PSS 传递函数:
$$G_{PSS}(s) = K_{stab} \\cdot \\frac{sT_w}{1+sT_w} \\cdot \\frac{(1+sT_1)(1+sT_3)}{(1+sT_2)(1+sT_4)}$$

In [ ]:
# ===== Ex 12.4: PSS 相位补偿设计 =====
# 无 PSS 时的特征值
r_no_pss = compute_heffron_phillips_constants(
    E_prime=1.2, V_infinity=1.0, X_total=0.5,
    Xd=1.8, Xd_prime=0.3, Xq=1.7,
    Td0_prime=8.0, H=5.0, D=0.0,
    delta_0=np.radians(30), Ka=50, Te=0.5, verbose=False
)

for ev, f in zip(r_no_pss.eigenvalues, r_no_pss.frequencies):
    if abs(ev.imag) > 1e-6 and f < 5.0:
        lambda_em = ev
        f_em = f
        break

print(f'机电模式: lambda = {lambda_em.real:.4f} + j{lambda_em.imag:.4f}')
print(f'  频率: f = {f_em:.2f} Hz')
print(f'  阻尼: zeta = {abs(lambda_em.real)/abs(lambda_em):.4f}')
print(f'  目标: zeta >= 0.05 (工程要求)')

# PSS 相位补偿设计
omega_em = abs(lambda_em.imag)  # 机电振荡角频率 (rad/s)
print(f'\nPSS 设计参数:')
print(f'  中心频率: omega_n = {omega_em:.2f} rad/s = {f_em:.2f} Hz')

# 超前环节 (补偿 AVR 相位滞后)
T1 = 0.15   # 超前时间常数
T2 = 0.03   # 滞后时间常数
Tw = 5.0    # 隔直时间常数 (washout)
Kstab = 10.0 # PSS 增益

# 计算 PSS 在 ω_em 处的相位超前
phi_lead1 = np.angle(1 + 1j*omega_em*T1) - np.angle(1 + 1j*omega_em*T2)
phi_washout = np.angle(1j*omega_em*Tw) - np.angle(1 + 1j*omega_em*Tw)
phi_pss = np.degrees(phi_lead1 + phi_washout)

print(f'\n  T1={T1:.2f}s, T2={T2:.2f}s, Tw={Tw:.1f}s, Kstab={Kstab:.1f}')
print(f'  PSS 相位补偿: {phi_pss:.1f}° lead (在 {f_em:.2f} Hz)')

# 幅频特性
freqs = np.logspace(-2, 1, 200)
omega = 2*np.pi*freqs
s = 1j*omega
pss_tf = Kstab * (s*Tw)/(1+s*Tw) * (1+s*T1)/(1+s*T2)

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(10, 8), sharex=True)
ax1.loglog(freqs, np.abs(pss_tf), 'b-', linewidth=2)
ax1.axvline(x=f_em, color='r', linestyle='--', label=f'f_em={f_em:.2f} Hz')
ax1.set_ylabel('Magnitude')
ax1.set_title('PSS Transfer Function Bode Plot')
ax1.grid(True, alpha=0.3)
ax1.legend()

ax2.semilogx(freqs, np.degrees(np.angle(pss_tf)), 'b-', linewidth=2)
ax2.axvline(x=f_em, color='r', linestyle='--', label=f'f_em={f_em:.2f} Hz')
ax2.set_xlabel('Frequency (Hz)')
ax2.set_ylabel('Phase (deg)')
ax2.grid(True, alpha=0.3)
ax2.legend()

plt.tight_layout()
plt.savefig('examples/kundur/pss_bode.png', dpi=150, bbox_inches='tight')
plt.show()

---
## 本章小结 (SMIB 小干扰稳定)

1. **Heffron-Phillips K1-K6 模型** 是分析 AVR 对阻尼影响的标准框架
2. **K5 < 0**（高功角时）是 AVR 负阻尼效应的根源
3. **AVR 增益 KA** 越大 → 负阻尼越严重（根轨迹向右半平面移动）
4. **PSS** 通过相位超前补偿抵消 AVR 负阻尼，将机电模式推向左半平面

> 下一章：[Ch.12b 两区域系统](./ch12b_two_area_small_signal.ipynb) — 多机系统小干扰稳定分析